# Replica della PCA varimax di Thiebaut de Schotten et al. 2020 sul disconnettoma voxelwise

**Domanda.** Se si applica al disconnettoma voxelwise `s2.4-vol` lo stesso metodo del paper (PCA sulla covarianza, poi rotazione varimax), quante componenti servono per spiegare la varianza, e quanto ogni componente dipende dalla dimensione della lesione?

**Cosa legge.** La matrice `matrix.npy` e `metadata.csv` della sessione `s2.4-vol` (5845 soggetti × 291.345 voxel) e la colonna `lesion_volume_voxels_2mm` di `assets/metadata/participants.csv`.

**Cosa scrive.** Solo una cache in `results/pipeline_building/thiebaut_2020_replication/<sessione>/` (ignorata da git), per non rifare ogni volta il calcolo. Il calcolo vero sta in `src/analysis/covariance_pca.py`.

> **Rispetto al paper.** Il notebook non cerca di riprodurre i numeri del paper: il paper usa 372 regioni, qui i voxel sono 291.345, quindi si confrontano le tendenze e si riporta quello che esce.

In [4]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import seaborn as sns
from nilearn import plotting

# La radice del repo è la prima cartella, risalendo da quella corrente, che contiene assets/metadata. Non dipende da
# quante volte si esegue la cella: l'os.chdir qui sotto sposta la cartella corrente nella radice.
_start = Path.cwd().resolve()
NOTEBOOK_REPO_ROOT = next((p for p in [_start, *_start.parents] if (p / "assets" / "metadata").is_dir()), None)
if NOTEBOOK_REPO_ROOT is None:
    raise FileNotFoundError(f"nessuna cartella con assets/metadata sopra {_start}: il notebook va aperto dentro il repo")
if str(NOTEBOOK_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_REPO_ROOT))
# load_participants_registry legge assets/metadata/participants.csv con un path relativo alla repo root.
os.chdir(NOTEBOOK_REPO_ROOT)

from src.analysis.covariance_pca import (
    CovariancePca,
    component_t_maps,
    fit_covariance_pca,
    map_reproducibility,
    pca_scores,
    positive_peak_signs,
    rotated_scores,
    squared_correlation,
)
from src.utils.participants import load_participants_registry

# src.analysis.* imposta matplotlib.use("Agg") al proprio import: senza questo i plot non compaiono.
%matplotlib inline

sns.set_theme(style="whitegrid")

## 1. Scelta della matrice e dei parametri

Questi sono i valori da modificare:

- `SESSION_DIR`: la cartella della matrice. Qui `s2.4-vol`, il disconnettoma voxelwise dei 5845 soggetti ammessi.

- `VARIANCE_SHARE_FOR_VARIMAX`: la quota di varianza da raggiungere per decidere quante componenti ruotare. Con `0.9` si ruotano le prime componenti che insieme spiegano almeno il 90%.

  > Nel paper (Risultati): *"30 components, out of 46 in total, explained more than 90% of the variance of the regions disconnected by a stroke (Fig. 2a)"*. Il paper non dichiara come ha scelto il numero di componenti: lo legge dalla curva della varianza cumulata. Qui si usa la stessa soglia come criterio.

- `VARIMAX_MAX_ITER`: iterazioni massime della rotazione. Qui 500.

  > Nel paper (Metodi, «Data compression»): *"a PCA using a covariance matrix and varimax rotation (with a maximum of 500 iterations for convergence)"*.

- `LESION_VOLUME_COLUMN`: la colonna del volume della lesione, in voxel a 2 mm.

- `USE_CACHE`: con `True` riusa il calcolo già fatto, con `False` lo rifà e sovrascrive la cache.

In [5]:
SESSION_DIR = Path("data/derived/sdc_matrix/07-10_s2.4-vol")
VARIANCE_SHARE_FOR_VARIMAX = 0.9
VARIMAX_MAX_ITER = 500
LESION_VOLUME_COLUMN = "lesion_volume_voxels_2mm"
USE_CACHE = True

CACHE_DIR = Path("results/pipeline_building/thiebaut_2020_replication") / SESSION_DIR.name

# La matrice resta su disco (mmap): ogni calcolo la legge a blocchi di colonne, mai tutta insieme.
X = np.load(SESSION_DIR / "matrix.npy", mmap_mode="r")
metadata = pd.read_csv(SESSION_DIR / "metadata.csv")
if len(metadata) != X.shape[0]:
    raise ValueError(f"metadata.csv ha {len(metadata)} righe ma la matrice ha {X.shape[0]} soggetti")

print(f"Sessione: {SESSION_DIR.name}")
print(f"Matrice: {X.shape[0]} soggetti x {X.shape[1]} voxel, tipo {X.dtype}")
print(f"Soggetti per dataset:")
for name, count in metadata["dataset"].value_counts().sort_index().items():
    print(f"  {name}: {count}")

Sessione: 07-10_s2.4-vol
Matrice: 5845 soggetti x 291345 voxel, tipo float32
Soggetti per dataset:
  UCL-UK/UCLStrokeData: 4119
  UKE/SFB936_ses01: 56
  UKE/WAKEUP_acute: 448
  UKLFR/stroke_UKLFR: 703
  UNIPD/NEMESIS_T0: 76
  UNIPD/PASPORT: 80
  UNIPD/PSP: 168
  UNIPD/WashU: 195


## 2. PCA sulla covarianza

**Cosa fa la PCA.** Trova nuovi assi, le **componenti**, lungo cui i soggetti differiscono di più.

- Ogni componente è una **combinazione lineare** dei voxel: una somma pesata.

- Le componenti sono **ordinate per varianza spiegata**: la prima conta più della seconda.

- Tenendo le prime *k* componenti, ogni soggetto è riassunto da *k* numeri invece che da 291.345 voxel, perdendo il meno possibile della variabilità.

**Cosa restituisce.**

| Oggetto | Cos'è | Dimensione |
|---|---|---|
| **Varianza spiegata** | quanta variabilità cattura ogni componente | una per componente |
| **Pesi** (*loadings*) | il peso di ogni voxel nella componente; uguali per tutti i soggetti | voxel × componenti |
| **Punteggi** (*score*) | la posizione di ogni soggetto sulla componente | soggetti × componenti |

Riportati sul cervello, i pesi di una componente ne formano la mappa dei pesi. Tecnicamente i pesi sono gli autovettori, e i *loadings* in senso stretto sono gli autovettori moltiplicati per la radice dell'autovalore: qui si usano come sinonimi.

**Il punteggio.** È la somma dei voxel di un soggetto, pesati con i pesi della componente. Dice quanto il pattern del soggetto si allinea alla mappa dei pesi, e con che intensità.

- Alto in valore assoluto: la mappa è molto presente nel soggetto.

- Il **segno** indica la direzione rispetto alla mappa. Il segno della componente stessa è arbitrario.

Esempio con 3 voxel (valori già centrati) e una componente con pesi (0,6; 0,0; 0,8):

| Soggetto | voxel₁ | voxel₂ | voxel₃ | Punteggio |
|---|---|---|---|---|
| 1 | 2 | −1 | 1 | 0,6·2 + 0,0·(−1) + 0,8·1 = **2,0** |
| 2 | −1 | 3 | −2 | 0,6·(−1) + 0,0·3 + 0,8·(−2) = **−2,2** |

I pesi sono uguali nelle due righe: cambiano i valori dei voxel, quindi cambia il punteggio. Un peso nullo (voxel₂) esclude quel voxel dalla componente. I soggetti 1 e 2 stanno ai due lati opposti della stessa componente.

**Covarianza, senza standardizzare.** Ogni voxel viene centrato (si toglie la sua media sui soggetti) ma non diviso per la sua deviazione standard. Un voxel che varia molto tra i soggetti pesa quindi di più di uno che varia poco.

**Perché la covarianza.** Ogni voxel è una probabilità di disconnessione, tutti sulla stessa scala: che un voxel vari più di un altro è informazione vera, non un accidente di unità di misura. Standardizzare darebbe lo stesso peso anche ai voxel quasi sempre a zero.

> Nel paper (Metodi, «Data compression»): *"Each matrix entered a PCA using a covariance matrix and varimax rotation (with a maximum of 500 iterations for convergence) in order to estimate the number of principal components to extract for the four conditions."* Il testo specifica solo «covariance matrix»: non dice altro sulla standardizzazione.

**Perché non è la PCA di sempre.** Il calcolo diretto costruirebbe una tabella che confronta i voxel tra loro: 291.345 × 291.345 valori, circa 680 GB. Non entra in memoria.

Ma i soggetti sono solo 5845. Si può costruire invece una tabella che confronta i **soggetti** tra loro (5845 × 5845, circa 270 MB). Le due tabelle hanno **gli stessi autovalori**, cioè la stessa varianza per componente, e dalla piccola si ricavano i pesi dei voxel.

Il risultato è la stessa PCA, calcolata per la strada più corta.

In [6]:
PCA_CACHE = CACHE_DIR / "pca.npz"
FINGERPRINT = CACHE_DIR / "source.json"

# La cache sa da quale matrice viene: se la matrice cambia, la cache non vale più e il notebook si ferma.
source = {
    "session": SESSION_DIR.name,
    "shape": list(X.shape),
    "matrix_mtime_ns": (SESSION_DIR / "matrix.npy").stat().st_mtime_ns,
}
if USE_CACHE and PCA_CACHE.is_file():
    if json.loads(FINGERPRINT.read_text()) != source:
        raise ValueError(f"la cache in {CACHE_DIR} viene da un'altra matrice: cancella la cartella o metti USE_CACHE = False")
    pca = CovariancePca.load(PCA_CACHE)
    print(f"PCA letta dalla cache: {PCA_CACHE}")
else:
    pca = fit_covariance_pca(X)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    pca.save(PCA_CACHE)
    FINGERPRINT.write_text(json.dumps(source))
    print(f"PCA calcolata e salvata in {PCA_CACHE}")

print(f"Componenti con varianza non nulla: {pca.n_nonzero_components()}")
print(f"Varianza totale della matrice: {pca.eigenvalues.sum():.1f}")

ValueError: la cache in results/pipeline_building/thiebaut_2020_replication/07-10_s2.4-vol viene da un'altra matrice: cancella la cartella o metti USE_CACHE = False

## 3. Quante componenti servono per il 50%, l'80%, il 90% della varianza

La **varianza cumulata** di *k* componenti è la somma delle quote delle prime *k*. Esempio: 26% + 25% = 51%, quindi con 2 componenti si supera il 50%.

La tabella dà, per ogni soglia, il numero minimo di componenti che la raggiunge.

> Paper: 30 componenti su 46 spiegano più del 90% (Fig. 2a).

In [ ]:
THRESHOLDS = (0.5, 0.8, 0.9)
n_for_share = {share: pca.n_components_for(share) for share in THRESHOLDS}

for share, n_components in n_for_share.items():
    print(f"Per spiegare almeno il {share:.0%} della varianza servono {n_components} componenti su {pca.n_nonzero_components()}")

print()
ratio = pca.explained_variance_ratio
cumulative = pca.cumulative_variance_ratio
for k in range(10):
    print(f"Componente {k + 1}: {ratio[k]:.1%} della varianza, {cumulative[k]:.1%} con le componenti fino a questa")

In [ ]:
N_SHOWN = 150
x = np.arange(1, N_SHOWN + 1)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, cumulative[:N_SHOWN], color="tab:blue")
for share, n_components in n_for_share.items():
    ax.axhline(share, color="grey", linestyle=":", linewidth=1)
    ax.plot([n_components], [cumulative[n_components - 1]], "o", color="tab:red")
    ax.annotate(f"{share:.0%}: {n_components} comp.", (n_components, cumulative[n_components - 1]),
                textcoords="offset points", xytext=(8, -14))
ax.set_xlabel("Numero di componenti")
ax.set_ylabel("Varianza cumulata")
ax.set_title(f"{SESSION_DIR.name}: varianza cumulata (prime {N_SHOWN} componenti)")
ax.set_ylim(0, 1.02)
plt.show()

## 4. Rotazione varimax

**Il problema.** Ogni componente PCA è un mix di tutto il cervello: difficile da leggere.

**Il varimax** ruota le prime *k* componenti, che restano ortogonali, finché ogni voxel pesa soprattutto su una componente. Ogni componente diventa un gruppo di voxel che si disconnettono insieme. La varianza totale delle *k* componenti non cambia: cambia solo come è divisa.

**I punteggi** si ricalcolano sui pesi ruotati. Il paper li ottiene per regressione multipla. Hanno tutti varianza 1.

In [ ]:
N_COMPONENTS = pca.n_components_for(VARIANCE_SHARE_FOR_VARIMAX)
print(f"Ruoto le prime {N_COMPONENTS} componenti (quelle che insieme spiegano almeno il {VARIANCE_SHARE_FOR_VARIMAX:.0%})")

VARIMAX_CACHE = CACHE_DIR / "varimax.npz"
varimax_source = {**source, "n_components": N_COMPONENTS, "max_iter": VARIMAX_MAX_ITER}
VARIMAX_FINGERPRINT = CACHE_DIR / "varimax_source.json"

if USE_CACHE and VARIMAX_CACHE.is_file():
    if json.loads(VARIMAX_FINGERPRINT.read_text()) != varimax_source:
        raise ValueError(f"la cache del varimax in {CACHE_DIR} ha altri parametri: metti USE_CACHE = False per rifarla")
    cached = np.load(VARIMAX_CACHE)
    scores, rotated_loadings = cached["scores"], cached["rotated_loadings"]
    print(f"Varimax letto dalla cache: {VARIMAX_CACHE}")
else:
    scores, rotated_loadings = rotated_scores(X, pca, N_COMPONENTS, rotation="varimax", max_iter=VARIMAX_MAX_ITER)
    np.savez(VARIMAX_CACHE, scores=scores, rotated_loadings=rotated_loadings)
    VARIMAX_FINGERPRINT.write_text(json.dumps(varimax_source))
    print(f"Varimax calcolato e salvato in {VARIMAX_CACHE}")

print(f"Punteggi: {scores.shape[0]} soggetti x {scores.shape[1]} componenti")
print(f"Pesi ruotati: {rotated_loadings.shape[0]} voxel x {rotated_loadings.shape[1]} componenti")

**Cosa si perde.** L'ordine per varianza. Si ricostruisce con la quota di varianza che ogni componente spiega: somma dei pesi al quadrato della sua colonna, divisa per la varianza totale.

**Attenzione.** Il varimax dà componenti dall'aspetto pulito anche se nei dati non ci sono gruppi. La verifica è la riproducibilità (sezione 6).

**Come si legge la tabella.** Ogni riga è un posto nella graduatoria per varianza. A sinistra la componente PCA, a destra la componente ruotata che lo occupa (*n.* è la sua etichetta nelle sezioni seguenti).

In [ ]:
# Quota di varianza spiegata da ogni componente ruotata: somma dei pesi al quadrato della sua colonna,
# divisa per la varianza totale. Prima della rotazione la stessa formula dà la quota della PCA.
rotated_share = (rotated_loadings**2).sum(axis=0) / pca.eigenvalues.sum()
by_variance = np.argsort(rotated_share)[::-1]

print(f"Varianza spiegata dalle {N_COMPONENTS} componenti: {rotated_share.sum():.1%} dopo la rotazione, "
      f"{pca.cumulative_variance_ratio[N_COMPONENTS - 1]:.1%} prima (il totale non cambia, cambia come è ripartita)")
print()
print(f"{'Posto':>6} | {'PCA':>8} | Componente ruotata")
for place, index in enumerate(by_variance[:10], start=1):
    print(f"{place:>5}º | {pca.explained_variance_ratio[place - 1]:>8.1%} | n. {index + 1:>2}, {rotated_share[index]:.1%}")

## 5. Quanto ogni componente dipende dal volume della lesione

**Domanda.** Una componente descrive una zona, o solo quanto è grande la lesione?

**R²** è la quota della variabilità del punteggio spiegata dal volume (il quadrato della correlazione).

- R² = 0.04: componente quasi indipendente dalla dimensione.

- R² = 0.50: componente che misura soprattutto la dimensione.

Si calcola sulle componenti ruotate e non ruotate.

> Paper: R² < 0.1 per tutte le componenti ruotate.

In [ ]:
registry = load_participants_registry().set_index("subject_id")
missing = sorted(set(metadata["subject_id"]) - set(registry.index))
if missing:
    raise ValueError(f"{len(missing)} soggetti della matrice non sono nel registro: {missing[:5]}")
volume = registry.loc[metadata["subject_id"], LESION_VOLUME_COLUMN].to_numpy(dtype=float)
if np.isnan(volume).any():
    raise ValueError(f"{int(np.isnan(volume).sum())} soggetti senza {LESION_VOLUME_COLUMN}")

r2_rotated = squared_correlation(scores, volume)
r2_unrotated = squared_correlation(pca_scores(X, pca, N_COMPONENTS), volume)

print(f"Volume della lesione ({LESION_VOLUME_COLUMN}): mediana {np.median(volume):.0f} voxel, da {volume.min():.0f} a {volume.max():.0f}")
print()
for name, r2 in (("ruotate", r2_rotated), ("non ruotate", r2_unrotated)):
    print(f"Componenti {name}: R² massimo {r2.max():.3f}, mediano {np.median(r2):.3f}, {int((r2 > 0.1).sum())} su {len(r2)} sopra 0.1")

print()
for k in np.argsort(r2_rotated)[::-1][:5]:
    print(f"Componente ruotata {k + 1}: R² con il volume = {r2_rotated[k]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
positions = np.arange(1, N_COMPONENTS + 1)
ax.bar(positions - 0.2, r2_rotated, width=0.4, label="ruotate (varimax)", color="tab:blue")
ax.bar(positions + 0.2, r2_unrotated, width=0.4, label="non ruotate (PCA)", color="tab:orange")
ax.axhline(0.1, color="tab:red", linestyle="--", linewidth=1, label="soglia 0.1 del paper")
ax.set_xlabel("Componente")
ax.set_ylabel("R² con il volume della lesione")
ax.set_title(f"{SESSION_DIR.name}: dipendenza dal volume della lesione")
ax.legend()
plt.show()

## 6. Mappe delle componenti nel cervello

**Mappa *t*.** Per ogni voxel si mette in relazione la disconnessione con i punteggi di tutte le componenti insieme (regressione multipla). Per ogni componente il coefficiente, diviso per il suo errore standard, dà *t*.

- *t* alto: effetto solido.

- *t* vicino a 0: nessun effetto.

- Il segno è la direzione.

I *t* di tutti i voxel formano la mappa della componente. È simile alla mappa dei pesi, ma tiene conto dell'incertezza di ogni voxel. Il paper usa `randomise` (1000 permutazioni); qui non c'è test di permutazione.

**Riproducibilità.** Si dividono i soggetti in due metà a caso, si rifà la regressione in ciascuna e si correlano (Pearson, sui voxel) le due mappe di ogni componente. 1 = uguali, 0 = niente in comune. I punteggi sono quelli di tutti i soggetti, quindi si verifica che la mappa si ritrovi, non che la PCA sia stabile.

> Paper: 0.813 ± 0.079, con metà da 666 soggetti. Qui sono circa 2900 per metà: i valori non sono direttamente confrontabili.

Valori da modificare:

- `COMPONENTS_TO_PLOT`: le componenti da disegnare, per esempio `[1, 2, 3, 4]`.

- `MAP_THRESHOLD_FRACTION`: la quota del picco sotto cui un voxel non è colorato (`0.3` = almeno il 30%).

- `SPLIT_SEED`: il seme della divisione in metà.

In [ ]:
COMPONENTS_TO_PLOT = [1, 2, 3, 4]
MAP_THRESHOLD_FRACTION = 0.3
SPLIT_SEED = 0

if not 0.0 <= MAP_THRESHOLD_FRACTION < 1.0:
    raise ValueError(f"MAP_THRESHOLD_FRACTION deve stare tra 0 e 1 (1 escluso), trovato {MAP_THRESHOLD_FRACTION}")
bad = [k for k in COMPONENTS_TO_PLOT if not 1 <= k <= N_COMPONENTS]
if bad or len(set(COMPONENTS_TO_PLOT)) != len(COMPONENTS_TO_PLOT):
    raise ValueError(f"COMPONENTS_TO_PLOT deve avere numeri distinti tra 1 e {N_COMPONENTS}, trovato {COMPONENTS_TO_PLOT}")

# Le due metà: sorteggio dei soggetti con un seme fisso, così il risultato si ripete.
order = np.random.default_rng(SPLIT_SEED).permutation(X.shape[0])
half_a = np.sort(order[: X.shape[0] // 2])
half_b = np.sort(order[X.shape[0] // 2 :])
print(f"Metà A: {len(half_a)} soggetti, metà B: {len(half_b)} soggetti (seme {SPLIT_SEED})")

MAPS_CACHE = CACHE_DIR / "maps.npz"
MAPS_FINGERPRINT = CACHE_DIR / "maps_source.json"
maps_source = {**varimax_source, "split_seed": SPLIT_SEED}

if USE_CACHE and MAPS_CACHE.is_file():
    if json.loads(MAPS_FINGERPRINT.read_text()) != maps_source:
        raise ValueError(f"la cache delle mappe in {CACHE_DIR} ha altri parametri: metti USE_CACHE = False per rifarla")
    cached = np.load(MAPS_CACHE)
    t_full, t_a, t_b = cached["t_full"], cached["t_a"], cached["t_b"]
    n_degenerate = cached["n_degenerate"]
    print(f"Mappe lette dalla cache: {MAPS_CACHE}")
else:
    maps_full, maps_a, maps_b = component_t_maps(X, scores, [np.arange(X.shape[0]), half_a, half_b])
    t_full, t_a, t_b = (m.t.astype(np.float32) for m in (maps_full, maps_a, maps_b))
    n_degenerate = np.array([m.n_degenerate_voxels for m in (maps_full, maps_a, maps_b)])
    np.savez(MAPS_CACHE, t_full=t_full, t_a=t_a, t_b=t_b, n_degenerate=n_degenerate)
    MAPS_FINGERPRINT.write_text(json.dumps(maps_source))
    print(f"Mappe calcolate e salvate in {MAPS_CACHE}")

print(f"Voxel senza varianza residua (t messo a 0): {n_degenerate[0]} su tutti i soggetti, {n_degenerate[1]} nella metà A, {n_degenerate[2]} nella metà B")

# Stessi segni per le due metà e per la mappa completa: la correlazione tra metà non dipende dal segno scelto.
signs = positive_peak_signs(t_full)
t_full, t_a, t_b = (t * signs[:, None] for t in (t_full, t_a, t_b))
reproducibility = map_reproducibility(t_a, t_b)

In [ ]:
print(f"Riproducibilità tra le due metà, media {reproducibility.mean():.3f} ± {reproducibility.std(ddof=1):.3f} (paper: 0.813 ± 0.079)")
print(f"Da {reproducibility.min():.3f} (componente {reproducibility.argmin() + 1}) a {reproducibility.max():.3f} (componente {reproducibility.argmax() + 1})")
print()
for k in range(N_COMPONENTS):
    print(f"Componente {k + 1}: riproducibilità = {reproducibility[k]:.3f}, t massimo in valore assoluto = {np.abs(t_full[k]).max():.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(np.arange(1, N_COMPONENTS + 1), reproducibility, color="tab:blue")
ax.axhline(0.813, color="tab:red", linestyle="--", linewidth=1, label="media del paper (0.813)")
ax.set_xlabel("Componente")
ax.set_ylabel("Riproducibilità tra le due metà")
ax.set_title(f"{SESSION_DIR.name}: riproducibilità delle mappe")
ax.set_ylim(0, 1.02)
ax.legend()
plt.show()

### Le mappe

Una figura per componente, con le sezioni sagittale, coronale e assiale sul picco di *t*. Sfondo: template MNI T1 a 1 mm.

**Come si leggono.**

| Cosa vedi | Significa |
|---|---|
| **Rosso** (*t* positivo) | più il soggetto ha questa componente, più quel voxel è disconnesso |
| **Blu** (*t* negativo) | più il soggetto ha questa componente, meno quel voxel è disconnesso |
| **Colore forte** (*t* alto in valore assoluto) | effetto solido |
| **Colore assente** | effetto sotto la soglia, o nessun effetto |

**Cosa si cerca.** Dove sta la zona rossa. Se è compatta, la componente è una zona del cervello.

**Segno.** È arbitrario: ogni mappa è orientata con il picco rosso, quindi si guarda la forma e non il colore.

**Soglia.** Si colorano solo i voxel con |*t*| almeno una frazione del picco di quella componente (`MAP_THRESHOLD_FRACTION`): una soglia fissa, come |*t*| > 3, colorerebbe quasi tutto il cervello. Il valore del picco è nel titolo di ogni figura.

In [ ]:
mask = np.load(SESSION_DIR / "non_constant_mask.npy")
# Geometria delle mappe: la griglia a 2 mm (solo intestazione). Sfondo: il T1 a 1 mm, perché il file T1 a 2 mm ha i dati troncati.
template = nib.load("assets/templates/tpl-MNI152NLin6Asym_res-2_T1w.nii.gz")
background = nib.load("assets/templates/tpl-MNI152NLin6Asym_res-1_T1w.nii.gz")
if mask.size != int(np.prod(template.shape)) or int(mask.sum()) != X.shape[1]:
    raise ValueError(f"la maschera ({mask.size} voxel, {int(mask.sum())} veri) non corrisponde al template {template.shape} o alla matrice ({X.shape[1]} colonne)")


def map_to_image(values):
    # Le colonne di X sono i voxel della maschera, nell'ordine in cui la griglia è stata srotolata (C-order).
    volume = np.zeros(mask.size, dtype=np.float32)
    volume[mask] = values
    return nib.Nifti1Image(volume.reshape(template.shape), template.affine)


for k in COMPONENTS_TO_PLOT:
    t_map = t_full[k - 1]
    peak_column = int(np.argmax(np.abs(t_map)))
    peak_voxel = np.unravel_index(np.flatnonzero(mask)[peak_column], template.shape)
    peak_mm = nib.affines.apply_affine(template.affine, peak_voxel)
    plotting.plot_stat_map(
        map_to_image(t_map),
        bg_img=background,
        threshold=MAP_THRESHOLD_FRACTION * abs(t_map[peak_column]),
        cut_coords=tuple(peak_mm),
        display_mode="ortho",
        title=f"Componente {k}: t massimo {t_map[peak_column]:.0f}, riproducibilità {reproducibility[k - 1]:.2f}",
    )
    plt.show()